# `Part 04` — Poetry's trajectories

With the model fit and validated in notebook 03, this notebook turns it into what it actually says about the PPA record: the three kinds of reception trajectory, the risers and decliners with uncertainty, and the literary periods in motion. A supporting author-level view is included near the end (not in the paper, but it's a natural aggregation of the same posterior).

Notebook reads the model bundle persisted by notebook 03 and **rebuilds the (unfitted) Bambi model** from the saved design frame, then loads the cached posterior — so nothing here re-samples. Run the setup cell first.

In [15]:
# --- Setup: load the fitted model from notebook 03 -------------------------
import sys, time as _time, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import altair as alt
import arviz as az
import bambi as bmb

for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        sys.path.insert(0, str(_p / "src"))
        break
else:
    raise FileNotFoundError(
        "Could not find src/paths.py — run this notebook with cwd = repository root"
    )

from paths import EXPORTS_DIR, MODEL_DIR, FIGURES_DIR, POETRY_METADATA_PATH

alt.data_transformers.disable_max_rows()

# corpus + model bundle
excerpts_df       = pl.read_parquet(EXPORTS_DIR / "excerpts_df.parquet")
_bambi_df         = pd.read_parquet(MODEL_DIR / "bambi_df.parquet")
_prev_df          = pd.read_parquet(MODEL_DIR / "prev_df.parquet")
_prev_model_df    = pd.read_parquet(MODEL_DIR / "prev_model_df.parquet")
_compare_df       = pd.read_parquet(MODEL_DIR / "compare_df.parquet")
_slope_info       = pd.read_parquet(MODEL_DIR / "slope_info.parquet")
_modeled_poems_pl = pl.read_parquet(MODEL_DIR / "modeled_poems.parquet")
with open(MODEL_DIR / "model_meta.json") as _f:
    _meta = _json.load(_f)
MODEL_YEAR_MIN = int(_meta["MODEL_YEAR_MIN"])
POEM_PERIOD = _meta["POEM_PERIOD"]
PREVALENCE_DEVIATION_SCALE = _meta["PREVALENCE_DEVIATION_SCALE"]
_bambi_formula = _meta["bambi_formula"]
RUN_BAMBI = True
_PLOT_FLOOR = 1e-6

# rebuild the (unfitted) model so .predict() works, then load the cached posterior
_bambi_model = bmb.Model(_bambi_formula, _bambi_df, family="beta_binomial")
_bambi_idata = az.from_netcdf(str(MODEL_DIR / "bambi_idata.nc"))
_posterior = _bambi_idata.posterior

# recover the per-poem random-slope variable name from the posterior
_candidates = []
for _v in _posterior.data_vars:
    if "poem_id" not in _v:
        continue
    if "I(" in _v or "**" in _v:
        continue
    if not _v.startswith("period_z") and "period_z|" not in _v:
        continue
    _extra_dims = [d for d in _posterior[_v].dims if d not in ("chain", "draw")]
    if _extra_dims:
        _candidates.append((_v, _extra_dims))
_candidates.sort(key=lambda x: len(x[0]))
_slope_var, _extra_dims = _candidates[0]
_poem_dim = _extra_dims[0]

# small helpers reused by the findings cells
def _safe_prob(values, eps: float = 1e-6):
    _arr = np.asarray(values, dtype=float)
    return np.clip(_arr, eps, 1 - eps)

def _safe_logit(values, eps: float = 1e-6):
    _p = _safe_prob(values, eps=eps)
    return np.log(_p / (1 - _p))

def _pretty_axis_label_expr() -> str:
    return "join(split(datum.label,','),'')"

def _poem_label_expr() -> "pl.Expr":
    _t = pl.col("poem_title").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars()
    _p = pl.col("poem_id").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars()
    return pl.when(_t != "").then(_t).otherwise(_p).alias("poem_label")

print(f"Loaded model bundle | poems: {_modeled_poems_pl.height} | slope var: {_slope_var}")

Loaded model bundle | poems: 906 | slope var: period_z|poem_id


## 1. Three kinds of reception trajectory

With the primary model fit and checked, we turn to what it actually says about the PPA record. Five visualisations, each answering a different question. §1.1 gives us a direct look at the top-quoted poems against both Bayesian and unpooled baselines. §1.2 strips the shared time curve out so each poem's own signal becomes visible. §1.3 picks poems *by archetype* — risers, flat canon, decliners — rather than by volume, to see the full range of trajectory shapes. §1.4 compresses 50 poems and 16 decades into a single heatmap that reads as a canon-shift story. §1.5 opens a handful of chosen poems into full trajectories, centered within each poem so we compare shape rather than absolute prominence.

### 1.1 The top-12 poems with shrinkage context

One panel per poem, for the twelve most-quoted poems in the modelled set. Each panel shows the observed proportion per decade (grey dots), the Bayesian posterior-mean fit (blue line) with its 95 % credible band, and the quasi-binomial fit (red dashed) for comparison. Where the Bayesian and QB curves diverge noticeably, the hierarchical shrinkage is doing real work — usually at the sparse early and late edges of a poem's at-risk window, where unpooled estimation would be overconfident. Where the two curves track each other closely, the poem has enough data to speak for itself.

In [16]:
if not RUN_BAMBI or "_bambi_idata" not in dir():
    print("Model bundle not loaded — run the setup cell at the top of this notebook.")
else:
    print("Extracting per-observation Bayesian fitted probabilities…")
    _t_pred = _time.time()
    _pp_post = _bambi_idata.posterior
    if "mu" not in _pp_post.data_vars and "p" not in _pp_post.data_vars:
        try:
            _bambi_model.predict(_bambi_idata, kind="response_params", inplace=True)
            _pp_post = _bambi_idata.posterior
        except Exception as _e_pred:
            print(f"  predict failed: {_e_pred}")
    _mu_var = None
    for _v in _pp_post.data_vars:
        if _v.endswith("_mean") or _v == "p" or _v == "mu":
            _mu_var = _v
            break
    if _mu_var is None:
        for _v in _pp_post.data_vars:
            _arr = _pp_post[_v]
            if any(d == "__obs__" for d in _arr.dims) or len(_arr.dims) >= 3:
                if "poem" not in _v and "author" not in _v and "sigma" not in _v and "kappa" not in _v:
                    _mu_var = _v
                    break
    if _mu_var is None:
        print("Could not identify fitted-probability posterior variable; listing candidates:")
        for _v in _pp_post.data_vars:
            print(f"  {_v}: {list(_pp_post[_v].dims)}")
        raise RuntimeError("Could not find fitted-probability posterior variable.")
    print(f"  using fitted-prob variable: {_mu_var}")

    _mu_array = _pp_post[_mu_var].stack(draws=("chain", "draw"))
    _obs_dim = [d for d in _mu_array.dims if d != "draws"][0]
    _mu_vals = np.asarray(_mu_array.values)
    if _mu_vals.shape[0] != len(_bambi_df):
        _mu_vals = _mu_vals.T

    _bayes_mean = _mu_vals.mean(axis=1)
    _bayes_lo = np.percentile(_mu_vals, 2.5, axis=1)
    _bayes_hi = np.percentile(_mu_vals, 97.5, axis=1)
    print(f"  posterior-mean compute in {_time.time() - _t_pred:.1f}s  "
          f"(shape: {_mu_vals.shape[0]} obs × {_mu_vals.shape[1]} draws)")

    _plot_df_b = _bambi_df.copy()
    _plot_df_b["p_bayes"] = _bayes_mean
    _plot_df_b["p_bayes_lo"] = _bayes_lo
    _plot_df_b["p_bayes_hi"] = _bayes_hi
    _plot_df_b = _plot_df_b.merge(
        _prev_model_df[["poem_id", "period", "p_fit"]].rename(columns={"p_fit": "p_qb"}),
        on=["poem_id", "period"],
        how="left",
    )

    _baseline_bayes_df = (
        _plot_df_b.groupby("period", as_index=False)
        .agg(baseline_p=("p_bayes", "mean"))
        .sort_values("period")
    )

    _FACET_N_B = 24
    _facet_ids_b = _modeled_poems_pl.head(_FACET_N_B).to_pandas()["poem_id"].tolist()
    _facet_labels_b = {
        row["poem_id"]: row["poem_label"][:40]
        for _, row in _modeled_poems_pl.head(_FACET_N_B).to_pandas().iterrows()
    }

    # Shared X axis for every panel — drop the thousands-comma (1,800 → 1800).
    _xax_period_p = alt.X(
        "period:Q",
        title="period",
        axis=alt.Axis(format=".0f", labelExpr=_pretty_axis_label_expr()),
    )

    _sm_panels_b = []
    for _fpid in _facet_ids_b:
        _pdf = _plot_df_b[_plot_df_b["poem_id"] == _fpid].copy()
        for _col in ["prop_quoted_raw", "p_bayes", "p_bayes_lo", "p_bayes_hi", "p_qb"]:
            if _col in _pdf.columns:
                _pdf[_col] = np.clip(_pdf[_col], _PLOT_FLOOR, 1 - _PLOT_FLOOR)
        _title_str = _facet_labels_b.get(_fpid, _fpid)[:40]

        _baseline_line = (
            alt.Chart(_baseline_bayes_df)
            .mark_line(strokeWidth=1.2, color="#555", strokeDash=[2, 3], opacity=0.55)
            .encode(x=_xax_period_p, y=alt.Y("baseline_p:Q", title="P(quoted)"))
        )
        _dots = (
            alt.Chart(_pdf)
            .mark_point(size=24, opacity=0.35, filled=True, color="#888")
            .encode(x=_xax_period_p, y="prop_quoted_raw:Q")
        )
        _band_b = (
            alt.Chart(_pdf)
            .mark_area(opacity=0.22, color="#4c78a8")
            .encode(x=_xax_period_p, y="p_bayes_lo:Q", y2="p_bayes_hi:Q")
        )
        _line_b = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=2.5, color="#4c78a8")
            .encode(x=_xax_period_p, y="p_bayes:Q")
        )
        _line_qb = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=1.8, color="#e45756", strokeDash=[4, 3])
            .encode(x=_xax_period_p, y="p_qb:Q")
        )
        _panel = (
            alt.layer(_baseline_line, _band_b, _dots, _line_b, _line_qb)
            .properties(width=170, height=120, title=alt.TitleParams(_title_str, fontSize=10))
        )
        _sm_panels_b.append(_panel)

    _rows_b = []
    for _r in range(0, _FACET_N_B, 4):
        _rows_b.append(alt.hconcat(*_sm_panels_b[_r : _r + 4]))

    from IPython.display import display as _display
    # Title and caption printed separately so you can copy them into a figure caption.
    _title_5i = f"Top {_FACET_N_B} poems — Bayesian vs quasi-binomial trajectories"
    _caption_5i = (
        "Blue = Bayesian posterior mean ± 95% CI | Red dashed = quasi-binomial | "
        "Grey dashed = cross-poem baseline (average fitted P per decade) | Grey dots = raw data."
    )
    print(f"Title:   {_title_5i}")
    print(f"Caption: {_caption_5i}")
    _display(alt.vconcat(*_rows_b))

Extracting per-observation Bayesian fitted probabilities…
  using fitted-prob variable: mu
  posterior-mean compute in 13.2s  (shape: 17278 obs × 8000 draws)
Title:   Top 24 poems — Bayesian vs quasi-binomial trajectories
Caption: Blue = Bayesian posterior mean ± 95% CI | Red dashed = quasi-binomial | Grey dashed = cross-poem baseline (average fitted P per decade) | Grey dots = raw data.


alt.VConcatChart(...)

### 1.2 Deviation from the shared baseline

The panels plot `P(quoted)` directly, so each panel inherits the shared `bs(period_c, df=4)` curve and the panels resemble each other. The peak in the mid-nineteenth century is real and shared across poems, but because it dominates every panel visually, it makes the individual-poem signal hard to see. This subsection changes one thing: the y-axis goes from `P(quoted)` to **log-odds deviation from the cross-poem baseline**:

$$\text{dev}(\text{poem}, t) \;=\; \operatorname{logit}\big(\hat P_{\text{Bayes}}(\text{poem}, t)\big) \;-\; \operatorname{logit}\big(\bar P(t)\big)$$

where $\bar P(t)$ is the mean fitted probability across all modelled poems in decade $t$. Subtracting the shared hump out of every panel leaves the poem-specific component behind.

Reading guide:

- **Dashed horizontal line at 0** — the poem is at the corpus average in that decade.
- **Above 0** — the poem is *more* quoted than the average modelled poem; **below 0** — less.
- **Dotted horizontal line per panel** — each poem's lifetime mean deviation. This tells you the poem's overall level (canonical or fringe) separately from its trajectory.
- **Slope of the blue line** — the poem's trajectory. Tilting up means gaining share of citation; tilting down means losing it.
- **Blue 95 % CI band** — posterior uncertainty on the deviation at each decade.
- **Grey dots** — raw per-decade deviation, $\operatorname{logit}(\text{observed share}) - \operatorname{logit}(\text{baseline})$.

With the shared hump gone, the twenty-four panels look genuinely different. Some sit mostly flat above zero (stable canon), some sweep up from below zero to above (late arrivals), some descend from above zero toward baseline (classics losing ground). Each panel now shows the poem's own story, not the corpus's.

In [17]:
if not RUN_BAMBI or "_plot_df_b" not in dir():
    print("§1.1 must run first — requires `_plot_df_b` in the namespace.")
else:
    # Log-odds deviation from the cross-poem baseline (logit p − logit baseline).
    _base_logit_df = _baseline_bayes_df.copy()
    _base_logit_df["baseline_logit"] = _safe_logit(_base_logit_df["baseline_p"])

    _dev_df = _plot_df_b.merge(_base_logit_df[["period", "baseline_logit"]], on="period", how="left")
    _dev_df["logit_dev"]    = _safe_logit(_dev_df["p_bayes"])    - _dev_df["baseline_logit"]
    _dev_df["logit_dev_lo"] = _safe_logit(_dev_df["p_bayes_lo"]) - _dev_df["baseline_logit"]
    _dev_df["logit_dev_hi"] = _safe_logit(_dev_df["p_bayes_hi"]) - _dev_df["baseline_logit"]
    _dev_df["logit_dev_raw"] = _safe_logit(_dev_df["prop_quoted_raw"]) - _dev_df["baseline_logit"]

    # Hardcoded y-domain: the top-24 by lifetime_works are all canonical poems
    # above the cross-poem baseline, so a symmetric [-cap, +cap] wastes half the
    # panel. If you later swap `_facet_ids_b` to include sub-baseline poems,
    # widen the lower bound here (or restore the percentile-cap heuristic).
    _DEV_Y_DOMAIN = (-0.5, 3.0)
    _dev_domain = list(_DEV_Y_DOMAIN)
    _dev_scale = alt.Scale(domain=_dev_domain, clamp=True)

    # Shared X axis for every panel — drop the thousands-comma (1,800 → 1800).
    _xax_period_d = alt.X(
        "period:Q",
        title="period",
        axis=alt.Axis(format=".0f", labelExpr=_pretty_axis_label_expr()),
    )

    _zero_rule = (
        alt.Chart(pd.DataFrame({"y": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], strokeWidth=1.1, opacity=0.55)
        .encode(y=alt.Y("y:Q", scale=_dev_scale))
    )

    _sm_panels_dev = []
    for _fpid in _facet_ids_b:
        _pdf = _dev_df[_dev_df["poem_id"] == _fpid].copy()
        _title_str = _facet_labels_b.get(_fpid, _fpid)[:40]
        _lifetime_mean = float(_pdf["logit_dev"].mean())

        _mean_rule = (
            alt.Chart(pd.DataFrame({"y": [_lifetime_mean]}))
            .mark_rule(color="#4c78a8", strokeDash=[2, 2], strokeWidth=1.0, opacity=0.65)
            .encode(y=alt.Y("y:Q", scale=_dev_scale))
        )
        _dots = (
            alt.Chart(_pdf)
            .mark_point(size=24, opacity=0.35, filled=True, color="#888")
            .encode(x=_xax_period_d, y=alt.Y("logit_dev_raw:Q", scale=_dev_scale))
        )
        _band = (
            alt.Chart(_pdf)
            .mark_area(opacity=0.22, color="#4c78a8")
            .encode(
                x=_xax_period_d,
                y=alt.Y("logit_dev_lo:Q", scale=_dev_scale,
                        title="log-odds dev. vs baseline"),
                y2=alt.Y2("logit_dev_hi:Q"),
            )
        )
        _line = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=2.5, color="#4c78a8")
            .encode(x=_xax_period_d,
                    y=alt.Y("logit_dev:Q", scale=_dev_scale,
                            title="log-odds dev. vs baseline"))
        )
        _panel = (
            alt.layer(_zero_rule, _mean_rule, _band, _dots, _line)
            .properties(width=170, height=120,
                        title=alt.TitleParams(_title_str, fontSize=10))
        )
        _sm_panels_dev.append(_panel)

    _rows_dev = [
        alt.hconcat(*_sm_panels_dev[_r : _r + 4])
        for _r in range(0, _FACET_N_B, 4)
    ]

    from IPython.display import display as _display
    # Title and caption printed separately so you can copy them into a figure caption.
    _title_5ip = (
        f"Top {_FACET_N_B} poems — Bayesian deviation from the cross-poem baseline "
        "(log-odds scale)"
    )
    _caption_5ip = (
        "Black dashed = baseline (at 0) | Blue dashed = poem's lifetime mean | "
        "Blue line = Bayesian posterior mean ± 95% CI | Grey dots = raw deviation."
    )
    print(f"Title:   {_title_5ip}")
    print(f"Caption: {_caption_5ip}")
    _display(alt.vconcat(*_rows_dev))


Title:   Top 24 poems — Bayesian deviation from the cross-poem baseline (log-odds scale)
Caption: Black dashed = baseline (at 0) | Blue dashed = poem's lifetime mean | Blue line = Bayesian posterior mean ± 95% CI | Grey dots = raw deviation.


alt.VConcatChart(...)

### 1.3 Three archetypes — risers, flat canon, decliners

Both §1.1 and §1.2 show the top 24 poems by raw lifetime volume, which produces a panel of canonical heavyweights — all above baseline, differing mainly in slope. That is useful but it exhausts only one register of the corpus. Here we pick 24 poems by **archetype** instead of by volume, so the panel reads three archetypes of reception trajectory, eight examples each:

- **Row 1 — Risers.** Poems with the strongest *credible* positive slope ($\text{slope}_{\text{Bayes}} > 0$, 95 % CI strictly above 0). Their share of host works climbs across the window.
- **Row 2 — Flat canon.** High-volume poems (lifetime ≥ 150 works) whose slope is credibly close to zero ($|\text{slope}_{\text{Bayes}}| \le 0.03$). These are the canonical workhorses whose trajectory is near-horizontal: they do not grow, they do not fade, they just keep being quoted.
- **Row 3 — Decliners.** Poems with the strongest credible negative slope. Their share of host works falls across the window.

Within each row the poems are sorted by the magnitude of the archetype criterion, so each row reads left-to-right from most archetypal to weakest member.

Plot mechanics are identical to §1.2 — log-odds deviation vs the cross-poem baseline, posterior-mean blue line with 95 % CI band, grey raw-deviation dots, dashed line at each poem's lifetime mean. Only the *selection* changes. The y-axis is widened so decliners dipping below baseline remain visible. A slope of +0.25 log-odds per decade, remember, is roughly a 28 % odds-ratio-per-decade gain in the probability of being quoted.

In [18]:
if not RUN_BAMBI or "_dev_df" not in dir() or "_compare_df" not in dir():
    print("§1.1 and §1.2 must run first — requires `_dev_df` and `_compare_df`.")
else:
    # Archetype-selection knobs. Keep at the top of the cell so future-you can tune.
    _N_PER_ARCHETYPE       = 4
    _MIN_LW_CANON          = 150     # min lifetime_works to qualify as "canonical"
    _MAX_ABS_SLOPE_CANON   = 0.03    # max |slope_bayes| to qualify as "flat"
    _TYPED_Y_DOMAIN        = (-2.5, 3.0)
    _PANEL_W, _PANEL_H     = 180, 120
    _TYPED_LABEL_MAXLEN    = 32

    # Row 1 — strongest credible risers
    _risers = (
        _compare_df
        .loc[(_compare_df["slope_bayes"] > 0) & (_compare_df["slope_bayes_lo"] > 0)]
        .sort_values("slope_bayes", ascending=False)
        .head(_N_PER_ARCHETYPE)
        .assign(archetype="riser")
    )
    # Row 2 — high-volume, near-flat
    _flat_canon = (
        _compare_df
        .loc[
            (_compare_df["lifetime_works"] >= _MIN_LW_CANON)
            & (_compare_df["slope_bayes"].abs() <= _MAX_ABS_SLOPE_CANON)
        ]
        .sort_values("lifetime_works", ascending=False)
        .head(_N_PER_ARCHETYPE)
        .assign(archetype="flat canon")
    )
    # Row 3 — strongest credible decliners (sort ascending so most negative comes first)
    _decliners = (
        _compare_df
        .loc[(_compare_df["slope_bayes"] < 0) & (_compare_df["slope_bayes_hi"] < 0)]
        .sort_values("slope_bayes", ascending=True)
        .head(_N_PER_ARCHETYPE)
        .assign(archetype="decliner")
    )
    _typed_sel = pd.concat([_risers, _flat_canon, _decliners], ignore_index=True)
    _typed_ids = _typed_sel["poem_id"].tolist()
    _typed_labels = dict(
        zip(_typed_sel["poem_id"], _typed_sel["poem_label"].str[:_TYPED_LABEL_MAXLEN])
    )

    print(
        f"Typed selection: {len(_risers)} risers | {len(_flat_canon)} flat canon | "
        f"{len(_decliners)} decliners  ({len(_typed_sel)} total)"
    )
    print(
        _typed_sel[["archetype", "poem_label", "slope_bayes",
                    "slope_bayes_lo", "slope_bayes_hi", "lifetime_works"]]
        .assign(
            slope_bayes=lambda d: d["slope_bayes"].round(3),
            slope_bayes_lo=lambda d: d["slope_bayes_lo"].round(3),
            slope_bayes_hi=lambda d: d["slope_bayes_hi"].round(3),
        )
        .to_string(index=False)
    )

    _xax_typed = alt.X(
        "period:Q",
        title="period",
        axis=alt.Axis(format=".0f", labelExpr=_pretty_axis_label_expr()),
    )
    _typed_scale = alt.Scale(domain=list(_TYPED_Y_DOMAIN), clamp=True)

    _zero_rule_t = (
        alt.Chart(pd.DataFrame({"y": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], strokeWidth=1.1, opacity=0.55)
        .encode(y=alt.Y("y:Q", scale=_typed_scale))
    )

    _panels_typed = []
    for _fpid in _typed_ids:
        _pdf = _dev_df[_dev_df["poem_id"] == _fpid].copy()
        _title_str = _typed_labels.get(_fpid, _fpid)
        _lifetime_mean = float(_pdf["logit_dev"].mean())

        _mean_rule = (
            alt.Chart(pd.DataFrame({"y": [_lifetime_mean]}))
            .mark_rule(color="#4c78a8", strokeDash=[2, 2], strokeWidth=1.0, opacity=0.65)
            .encode(y=alt.Y("y:Q", scale=_typed_scale))
        )
        _dots = (
            alt.Chart(_pdf)
            .mark_point(size=22, opacity=0.35, filled=True, color="#888")
            .encode(x=_xax_typed, y=alt.Y("logit_dev_raw:Q", scale=_typed_scale))
        )
        _band = (
            alt.Chart(_pdf)
            .mark_area(opacity=0.22, color="#4c78a8")
            .encode(
                x=_xax_typed,
                y=alt.Y("logit_dev_lo:Q", scale=_typed_scale,
                        title="log-odds dev. vs baseline"),
                y2=alt.Y2("logit_dev_hi:Q"),
            )
        )
        _line = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=2.5, color="#4c78a8")
            .encode(x=_xax_typed,
                    y=alt.Y("logit_dev:Q", scale=_typed_scale,
                            title="log-odds dev. vs baseline"))
        )
        _panel = (
            alt.layer(_zero_rule_t, _mean_rule, _band, _dots, _line)
            .properties(width=_PANEL_W, height=_PANEL_H,
                        title=alt.TitleParams(_title_str, fontSize=9))
        )
        _panels_typed.append(_panel)

    # One archetype per row — 8 panels wide.
    _rows_typed = [
        alt.hconcat(*_panels_typed[_r : _r + _N_PER_ARCHETYPE])
        for _r in range(0, len(_panels_typed), _N_PER_ARCHETYPE)
    ]

    from IPython.display import display as _display
    # Title and caption printed separately so you can copy them into a figure caption.
    _title_5ipp = (
        f"Archetype small-multiples — {_N_PER_ARCHETYPE} risers | "
        f"{_N_PER_ARCHETYPE} flat canon | {_N_PER_ARCHETYPE} decliners"
    )
    _caption_5ipp = (
        "Row 1 = credible risers (sorted by slope desc). "
        "Row 2 = high-volume flat-trend poems (sorted by lifetime works). "
        "Row 3 = credible decliners (sorted by slope asc). "
        "Blue line = posterior mean ± 95% CI | Black dashed = cross-poem baseline | "
        "Blue dashed = poem's lifetime mean."
    )
    print(f"Title:   {_title_5ipp}")
    print(f"Caption: {_caption_5ipp}")
    _display(alt.vconcat(*_rows_typed))


Typed selection: 4 risers | 4 flat canon | 4 decliners  (12 total)
 archetype                                                                         poem_label  slope_bayes  slope_bayes_lo  slope_bayes_hi  lifetime_works
     riser                                  THE RIME OF THE ANCYENT MARINERE, IN SEVEN PARTS.        0.348           0.285           0.413             287
     riser                                                                 CCCI THE DAFFODILS        0.305           0.210           0.399             109
     riser CCCXXXVIII ODE ON INTIMATIONS OF IMMORTALITY FROM RECOLLECTIONS OF EARLY CHILDHOOD        0.296           0.235           0.360             263
     riser                                                             TALES OF A WAYSIDE INN        0.292           0.220           0.367             217
flat canon                                                                             Psalms       -0.013          -0.030           0.004            1110
fla

alt.VConcatChart(...)

### 1.4 The poem × decade heatmap

Fifty poems and sixteen decades compressed into a single chart. Each row is one of the top 50 poems, sorted by the decade in which its fitted prevalence peaked; each column is a decade; colour shows how far above or below the cross-poem baseline that poem sat in that decade, on the log-odds scale. Red cells are above baseline, blue cells are below, white cells are approximately at baseline. Empty cells correspond to decades before the poem's at-risk start year and are excluded by the left-truncation. The heatmap is the single most compressed view of the corpus-wide story: tilting the rows by their peak decade produces a diagonal band that reads left-to-right as the succession of canon phases over two centuries.

In [19]:
if not RUN_BAMBI or "_plot_df_b" not in dir():
    print("Bambi small-multiples (§1.1) must run first — required for _plot_df_b.")
else:
    from IPython.display import display as _display

    _heat_pool = _plot_df_b.copy()
    _bambi_baseline = (
        _heat_pool.groupby("period", as_index=False)
        .agg(baseline_fit=("p_bayes", "mean"))
        .sort_values("period")
    )
    _heat_pool = _heat_pool.merge(_bambi_baseline, on="period", how="left")

    _HEATMAP_N = 50
    _heat_ids = set(_modeled_poems_pl.head(_HEATMAP_N).to_pandas()["poem_id"])
    _heat_df = _heat_pool[_heat_pool["poem_id"].isin(_heat_ids)].copy()

    _heat_df["log_odds_dev"] = _safe_logit(_heat_df["p_bayes"]) - _safe_logit(_heat_df["baseline_fit"])
    _heat_df["poem_label_short"] = _heat_df["poem_label"].str[:45]

    _peak_idx = _heat_df.groupby("poem_label_short")["p_bayes"].idxmax()
    _peak_period_by_label = _heat_df.loc[_peak_idx, ["poem_label_short", "period"]].rename(
        columns={"period": "peak_period"}
    )
    _heat_df = _heat_df.merge(_peak_period_by_label, on="poem_label_short", how="left")

    _sort_order = (
        _heat_df.groupby("poem_label_short", as_index=False)
        .agg(peak_period=("peak_period", "first"), lifetime_works=("lifetime_works", "first"))
        .sort_values(["peak_period", "lifetime_works"], ascending=[True, False])
        ["poem_label_short"].tolist()
    )

    _lod_cap = float(_heat_df["log_odds_dev"].abs().quantile(0.95))
    _lod_cap = max(_lod_cap, 0.5)

    _heatmap = (
        alt.Chart(_heat_df)
        .mark_rect()
        .encode(
            x=alt.X("period:O", title="Decade", axis=alt.Axis(labelAngle=-45)),
            y=alt.Y("poem_label_short:N", title=None, sort=_sort_order,
                     axis=alt.Axis(labelLimit=280)),
            color=alt.Color(
                "log_odds_dev:Q",
                title="Log-odds vs baseline",
                scale=alt.Scale(scheme="redblue", domainMid=0, domain=[-_lod_cap, _lod_cap]),
            ),
            tooltip=[
                "poem_label_short",
                "period:O",
                alt.Tooltip("p_bayes:Q", format=".4f", title="fitted P (Bayes)"),
                alt.Tooltip("baseline_fit:Q", format=".4f", title="baseline"),
                alt.Tooltip("log_odds_dev:Q", format=".2f", title="log-odds dev"),
                alt.Tooltip("quoted_works:Q", title="quoted works"),
                alt.Tooltip("n_works:Q", title="host works"),
            ],
        )
        .properties(
            width=560,
            height=max(14 * _HEATMAP_N, 400),
            title=alt.TitleParams(
                f"Poem × decade prevalence heatmap — top {_HEATMAP_N} poems (Bayesian fit, sorted by peak decade)",
                subtitle="Red = above baseline | Blue = below baseline | White = at baseline",
            ),
        )
    )
    _display(_heatmap)


alt.Chart(...)

### 1.5 Selected poem trajectories — centered within poem

Eight poems chosen for the paper figure. Each panel plots decade-level deviation from that poem's own long-run mean in the modelled window, so the dashed horizontal line marks its average position and the panels become comparable on shape rather than absolute prominence. Grey points are observed deviations; the orange line is a smoothed empirical trajectory; the blue line is the regularised Bayesian fit with a 95 % credible band. A vertical dotted rule marks the poem's publication date. Trajectories are masked before that date.

In [20]:
if not RUN_BAMBI or "_dev_df" not in dir():
    print("§1.1 and §1.2 must run first — requires `_dev_df`.")
else:
    _SELECTED_POEMS = [
        {
            "poem_id": "Geoffrey-Chaucer_The-Canterbury-Tales",
            "label": "The Canterbury Tales",
            "author": "Geoffrey Chaucer",
            "exists_from": 1400,
            "date_label": "c. 1400",
        },
        {
            "poem_id": "Z200437755",
            "label": "Paradise Lost",
            "author": "John Milton",
            "exists_from": 1667,
            "date_label": "1667",
        },
        {
            "poem_id": "Z400343114",
            "label": "Pastoral V",
            "author": "Virgil / John Dryden",
            "exists_from": 1697,
            "date_label": "transl. 1697",
        },
        {
            "poem_id": "Z300509658",
            "label": "Winter",
            "author": "James Thomson",
            "exists_from": 1726,
            "date_label": "1726",
        },
        {
            "poem_id": "Z200439011",
            "label": "Elegy Written in a Country Church Yard",
            "author": "Thomas Gray",
            "exists_from": 1751,
            "date_label": "1751",
        },
        {
            "poem_id": "Z200379924",
            "label": "The Deserted Village",
            "author": "Oliver Goldsmith",
            "exists_from": 1770,
            "date_label": "1770",
        },
        {
            "poem_id": "Z200543922",
            "label": "Rime of the Ancyent Marinere",
            "author": "Samuel Taylor Coleridge",
            "exists_from": 1798,
            "date_label": "1798",
        },
        {
            "poem_id": "Z200628404",
            "label": "The Lay of the Last Minstrel",
            "author": "Walter Scott",
            "exists_from": 1805,
            "date_label": "1805",
        },
    ]

    SHOW_EMPIRICAL_LINE = True
    SHOW_EXISTENCE_RULE = True
    SAVE_FIG7 = True

    _EMP_LINE_OPACITY = 0.48
    _PANEL_W, _PANEL_H = 245, 150
    _LABEL_MAXLEN = 42
    _Y_DOMAIN = (-2, 2)
    _X_DOMAIN = [1690, 1920]
    _EMP_ROLLING_WINDOW = 3

    _selected_ids = [p["poem_id"] for p in _SELECTED_POEMS]
    _label_lookup = {p["poem_id"]: p["label"] for p in _SELECTED_POEMS}
    _author_lookup = {p["poem_id"]: p["author"] for p in _SELECTED_POEMS}
    _exists_lookup = {p["poem_id"]: p["exists_from"] for p in _SELECTED_POEMS}
    _date_lookup = {p["poem_id"]: p["date_label"] for p in _SELECTED_POEMS}

    _available_ids = set(_dev_df["poem_id"].unique())
    _missing = [pid for pid in _selected_ids if pid not in _available_ids]

    if _missing:
        print("Missing poem_id(s) in _dev_df:")
        for pid in _missing:
            print("  ", pid)
        print("\nThese may have been excluded by the support threshold or blocklist.")

    _plot_ids = [pid for pid in _selected_ids if pid in _available_ids]

    _sel_plot_df = _dev_df[_dev_df["poem_id"].isin(_plot_ids)].copy()
    _sel_plot_df["panel_label"] = _sel_plot_df["poem_id"].map(_label_lookup)
    _sel_plot_df["panel_label"] = _sel_plot_df["panel_label"].fillna(_sel_plot_df["poem_label"])
    _sel_plot_df["panel_label_short"] = _sel_plot_df["panel_label"].str[:_LABEL_MAXLEN]

    _baseline_emp_selected = (
        _dev_df.groupby("period", as_index=False)
        .agg(
            total_quoted=("quoted_works", "sum"),
            total_possible=("n_works", "sum"),
            n_cells=("poem_id", "count"),
        )
    )
    _baseline_emp_selected["baseline_p_emp"] = (
        (_baseline_emp_selected["total_quoted"] + 0.5 * _baseline_emp_selected["n_cells"])
        / (_baseline_emp_selected["total_possible"] + 1.0 * _baseline_emp_selected["n_cells"])
    )
    _baseline_emp_selected["baseline_logit_emp"] = _safe_logit(
        _baseline_emp_selected["baseline_p_emp"]
    )

    _x_scale = alt.Scale(domain=_X_DOMAIN, nice=False)
    _y_scale = alt.Scale(domain=list(_Y_DOMAIN), clamp=True)

    _x_axis = alt.X(
        "period:Q",
        title="Decade",
        scale=_x_scale,
        axis=alt.Axis(
            values=[1700, 1750, 1800, 1850, 1900],
            format=".0f",
            labelExpr=_pretty_axis_label_expr(),
        ),
    )

    _panels = []
    for _pid in _plot_ids:
        _pdf_full = _sel_plot_df[_sel_plot_df["poem_id"] == _pid].copy()
        _pdf_full = _pdf_full.sort_values("period").reset_index(drop=True)

        _title = _pdf_full["panel_label_short"].iloc[0]
        _author = _author_lookup.get(_pid, "")
        _date_label = _date_lookup.get(_pid, "")
        _subtitle = f"{_author} ({_date_label})" if _author or _date_label else ""

        _exists_from = _exists_lookup.get(_pid)
        _exists_period = None
        if _exists_from is not None:
            _exists_period = (_exists_from // 10) * 10
            _pdf = _pdf_full[_pdf_full["period"] >= _exists_period].copy()
        else:
            _pdf = _pdf_full.copy()

        if _pdf.empty:
            print(f"No plotted rows remain for {_pid} after existence-date masking.")
            continue

        _poem_mean = float(_pdf["logit_dev"].mean())
        _pdf["logit_dev_centered"] = _pdf["logit_dev"] - _poem_mean
        _pdf["logit_dev_lo_centered"] = _pdf["logit_dev_lo"] - _poem_mean
        _pdf["logit_dev_hi_centered"] = _pdf["logit_dev_hi"] - _poem_mean
        _pdf["logit_dev_raw_centered"] = _pdf["logit_dev_raw"] - _poem_mean

        _pdf["p_emp_smooth"] = (
            (_pdf["quoted_works"].astype(float) + 0.5)
            / (_pdf["n_works"].astype(float) + 1.0)
        )
        _pdf = _pdf.merge(
            _baseline_emp_selected[["period", "baseline_logit_emp"]],
            on="period",
            how="left",
        )
        _pdf["logit_emp"] = _safe_logit(_pdf["p_emp_smooth"])
        _pdf["logit_emp_dev"] = _pdf["logit_emp"] - _pdf["baseline_logit_emp"]
        _pdf["logit_emp_dev_smooth"] = (
            _pdf["logit_emp_dev"]
            .rolling(_EMP_ROLLING_WINDOW, center=True, min_periods=1)
            .mean()
        )
        _pdf["logit_emp_dev_smooth_centered"] = _pdf["logit_emp_dev_smooth"] - _poem_mean

        _poem_mean_rule = (
            alt.Chart(pd.DataFrame({"y": [0.0]}))
            .mark_rule(color="black", strokeDash=[4, 3], strokeWidth=1.1, opacity=0.65)
            .encode(y=alt.Y("y:Q", scale=_y_scale))
        )

        _layers = [_poem_mean_rule]

        if SHOW_EXISTENCE_RULE and _exists_period is not None and _exists_period >= _X_DOMAIN[0]:
            _exists_rule = (
                alt.Chart(pd.DataFrame({"x": [_exists_period]}))
                .mark_rule(color="#999", strokeDash=[2, 3], strokeWidth=1.0, opacity=0.45)
                .encode(x=alt.X("x:Q", scale=_x_scale))
            )
            _layers.append(_exists_rule)

        _raw_dots = (
            alt.Chart(_pdf)
            .mark_point(size=24, opacity=0.35, filled=True, color="#777")
            .encode(
                x=_x_axis,
                y=alt.Y(
                    "logit_dev_raw_centered:Q",
                    scale=_y_scale,
                    title="Deviation from poem's mean",
                ),
                tooltip=[
                    "panel_label:N",
                    "period:Q",
                    "quoted_works:Q",
                    "n_works:Q",
                    alt.Tooltip("prop_quoted_raw:Q", format=".3f"),
                    alt.Tooltip("logit_dev_raw_centered:Q", format=".2f"),
                    alt.Tooltip("logit_dev_centered:Q", format=".2f"),
                    alt.Tooltip("logit_dev:Q", title="absolute logit dev", format=".2f"),
                ],
            )
        )

        _band = (
            alt.Chart(_pdf)
            .mark_area(opacity=0.18, color="#4c78a8")
            .encode(
                x=_x_axis,
                y=alt.Y(
                    "logit_dev_lo_centered:Q",
                    scale=_y_scale,
                    title="Deviation from poem's mean",
                ),
                y2=alt.Y2("logit_dev_hi_centered:Q"),
            )
        )

        _bayes_line = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=2.4, color="#4c78a8")
            .encode(
                x=_x_axis,
                y=alt.Y(
                    "logit_dev_centered:Q",
                    scale=_y_scale,
                    title="Deviation from poem's mean",
                ),
            )
        )

        _emp_line = (
            alt.Chart(_pdf)
            .mark_line(strokeWidth=2.4, color="#c45200", opacity=_EMP_LINE_OPACITY)
            .encode(
                x=_x_axis,
                y=alt.Y(
                    "logit_emp_dev_smooth_centered:Q",
                    scale=_y_scale,
                    title="Deviation from poem's mean",
                ),
            )
        )

        _layers.extend([_band, _raw_dots, _bayes_line])
        if SHOW_EMPIRICAL_LINE:
            _layers.append(_emp_line)

        _panel = (
            alt.layer(*_layers)
            .properties(
                width=_PANEL_W,
                height=_PANEL_H,
                title=alt.TitleParams(
                    text=_title,
                    subtitle=_subtitle,
                    fontSize=11,
                    fontStyle="italic",
                    subtitleFontSize=9,
                    subtitleFontStyle="normal",
                    subtitleColor="#555",
                ),
            )
        )
        _panels.append(_panel)

    _rows = [
        alt.hconcat(*_panels[i : i + 2])
        for i in range(0, len(_panels), 2)
    ]

    _title_fig7 = "Selected Poem Trajectories, Centered Within Poem"
    _caption_fig7 = (
        "All panels share the same x-axis and grid. "
        "Trajectories are masked before each poem's publication/existence date. "
        "Blue line = Bayesian posterior mean ± 95% credible interval, centered around each poem's own visible-window mean. "
        + ("Orange line = smoothed empirical deviation, centered the same way. " if SHOW_EMPIRICAL_LINE else "")
        + "Grey dots = raw observed deviation. "
        "Black dashed = poem's own average fitted position. "
        + ("Grey vertical rule = poem publication/existence decade. " if SHOW_EXISTENCE_RULE else "")
        + "Values above zero mark decades when the poem sits above its own long-run level; "
        "values below zero mark decades when it sits below it."
    )

    print(f"Title:   {_title_fig7}")
    print(f"Caption: {_caption_fig7}")

    from IPython.display import display as _display

    if _rows:
        _chart = alt.vconcat(*_rows)
        _display(_chart)

        if SAVE_FIG7:
            _FIG_DIR = FIGURES_DIR
            _FIG_DIR.mkdir(parents=True, exist_ok=True)

            _chart.save(str(_FIG_DIR / "fig7_selected_poem_centered_trajectories.html"))
            _chart.save(str(_FIG_DIR / "fig7_selected_poem_centered_trajectories.svg"))
            try:
                _chart.save(str(_FIG_DIR / "fig7_selected_poem_centered_trajectories.pdf"))
            except Exception as _e:
                print("PDF save skipped:", _e)

            print(f"\nSaved figure to {_FIG_DIR}")
    else:
        print("No selected poems available in _dev_df.")

Title:   Selected Poem Trajectories, Centered Within Poem
Caption: All panels share the same x-axis and grid. Trajectories are masked before each poem's publication/existence date. Blue line = Bayesian posterior mean ± 95% credible interval, centered around each poem's own visible-window mean. Orange line = smoothed empirical deviation, centered the same way. Grey dots = raw observed deviation. Black dashed = poem's own average fitted position. Grey vertical rule = poem publication/existence decade. Values above zero mark decades when the poem sits above its own long-run level; values below zero mark decades when it sits below it.


alt.VConcatChart(...)


Saved figure to /Users/wouter/gitrepos/quotable_canon/exports/figures


## 2. Risers and decliners with uncertainty

*Which poems most clearly gain ground across the window, and which lose it?* This is the core quantitative finding of the notebook. For each poem we use its Bayesian posterior-mean slope — the log-odds change per decade on top of the shared corpus trend — and show a 95 % credible interval as a black error bar. Coloured bars are poems whose credible interval excludes zero (we can distinguish their trend from the average); faded grey bars are not statistically distinguishable from the average, even if their point estimate looks large.

A slope of +0.25 log-odds per decade is an odds ratio of $e^{0.25} \approx 1.28$ — roughly a 28 % per-decade increase in the odds that a host work quotes the poem, relative to the shared trend. A slope of −0.20 is an odds ratio of $e^{-0.20} \approx 0.82$ — an 18 % per-decade decrease.

In [21]:
if not RUN_BAMBI or "_bambi_idata" not in dir():
    print("Model bundle not loaded — run the setup cell at the top of this notebook.")
else:
    _bayes_rank = _compare_df.copy()
    _bayes_rank["bayes_significant"] = (
        (_bayes_rank["slope_bayes_lo"] > 0) | (_bayes_rank["slope_bayes_hi"] < 0)
    )
    _bayes_rank = _bayes_rank.sort_values("slope_bayes", ascending=False).reset_index(drop=True)

    _n_bayes_sig = int(_bayes_rank["bayes_significant"].sum())
    _n_qb_sig = int(_slope_info["significant"].sum())
    _delta_sig = _n_bayes_sig - _n_qb_sig
    print(f"Quasi-binomial slopes significant at 95%: {_n_qb_sig} / {len(_slope_info)}")
    print(f"Bayesian credible slopes excluding 0    : {_n_bayes_sig} / {len(_bayes_rank)}")
    if _delta_sig > 0:
        print(f"  → Bayesian partial pooling yields {_delta_sig} more credible nonzero slopes.")
    elif _delta_sig < 0:
        print(f"  → Bayesian shrinkage pushes {-_delta_sig} poems out of significance.")
    else:
        print("  → Bayesian and quasi-binomial agree on the number of nonzero slopes.")

    _bayes_disp = [
        "poem_label", "slope_bayes", "slope_bayes_lo", "slope_bayes_hi",
        "bayes_significant", "slope_hat", "lifetime_works",
    ]
    print("\nTop 12 Bayesian risers (posterior-mean slope, log-odds per decade):")
    print(
        _bayes_rank.head(12)[_bayes_disp]
        .assign(
            slope_bayes=lambda d: d["slope_bayes"].round(3),
            slope_bayes_lo=lambda d: d["slope_bayes_lo"].round(3),
            slope_bayes_hi=lambda d: d["slope_bayes_hi"].round(3),
            slope_hat=lambda d: d["slope_hat"].round(3),
        )
        .to_string(index=False)
    )
    print("\nTop 12 Bayesian decliners:")
    print(
        _bayes_rank.tail(12).sort_values("slope_bayes")[_bayes_disp]
        .assign(
            slope_bayes=lambda d: d["slope_bayes"].round(3),
            slope_bayes_lo=lambda d: d["slope_bayes_lo"].round(3),
            slope_bayes_hi=lambda d: d["slope_bayes_hi"].round(3),
            slope_hat=lambda d: d["slope_hat"].round(3),
        )
        .to_string(index=False)
    )

    _risers_b = _bayes_rank.head(15).copy()
    _decliners_b = _bayes_rank.tail(15).sort_values("slope_bayes").copy()
    _risers_b["poem_label_short"] = _risers_b["poem_label"].str[:45]
    _decliners_b["poem_label_short"] = _decliners_b["poem_label"].str[:45]

    _rise_b = (
        alt.Chart(_risers_b)
        .mark_bar()
        .encode(
            x=alt.X("slope_bayes:Q", title="Bayesian posterior-mean slope (log-odds per decade)"),
            y=alt.Y("poem_label_short:N", sort=None, title=None, axis=alt.Axis(labelLimit=320)),
            color=alt.condition(
                alt.datum.bayes_significant, alt.value("#c45200"), alt.value("#d3d3d3"),
            ),
            tooltip=[
                "poem_label",
                alt.Tooltip("slope_bayes:Q", format=".3f"),
                alt.Tooltip("slope_bayes_lo:Q", format=".3f"),
                alt.Tooltip("slope_bayes_hi:Q", format=".3f"),
                alt.Tooltip("slope_hat:Q", format=".3f", title="quasi-binomial"),
                "lifetime_works:Q",
            ],
        )
        .properties(width=330, height=340, title="Top 15 Bayesian risers")
    )
    _rise_b_err = (
        alt.Chart(_risers_b)
        .mark_rule(strokeWidth=1.5)
        .encode(
            x="slope_bayes_lo:Q", x2="slope_bayes_hi:Q",
            y=alt.Y("poem_label_short:N", sort=None),
        )
    )
    _decline_b = (
        alt.Chart(_decliners_b)
        .mark_bar()
        .encode(
            x=alt.X("slope_bayes:Q", title="Bayesian posterior-mean slope (log-odds per decade)"),
            y=alt.Y("poem_label_short:N", sort=None, title=None, axis=alt.Axis(labelLimit=320)),
            color=alt.condition(
                alt.datum.bayes_significant, alt.value("#154e8a"), alt.value("#d3d3d3"),
            ),
            tooltip=[
                "poem_label",
                alt.Tooltip("slope_bayes:Q", format=".3f"),
                alt.Tooltip("slope_bayes_lo:Q", format=".3f"),
                alt.Tooltip("slope_bayes_hi:Q", format=".3f"),
                alt.Tooltip("slope_hat:Q", format=".3f", title="quasi-binomial"),
                "lifetime_works:Q",
            ],
        )
        .properties(width=330, height=340, title="Top 15 Bayesian decliners")
    )
    _decline_b_err = (
        alt.Chart(_decliners_b)
        .mark_rule(strokeWidth=1.5)
        .encode(
            x="slope_bayes_lo:Q", x2="slope_bayes_hi:Q",
            y=alt.Y("poem_label_short:N", sort=None),
        )
    )
    _xref_b = (
        alt.Chart(pd.DataFrame({"x": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], opacity=0.55)
        .encode(x="x:Q")
    )

    from IPython.display import display as _display
    _display(alt.hconcat(
        alt.layer(_rise_b, _rise_b_err, _xref_b),
        alt.layer(_decline_b, _decline_b_err, _xref_b),
    ))

Quasi-binomial slopes significant at 95%: 373 / 906
Bayesian credible slopes excluding 0    : 401 / 906
  → Bayesian partial pooling yields 28 more credible nonzero slopes.

Top 12 Bayesian risers (posterior-mean slope, log-odds per decade):
                                                                        poem_label  slope_bayes  slope_bayes_lo  slope_bayes_hi  bayes_significant  slope_hat  lifetime_works
                                 THE RIME OF THE ANCYENT MARINERE, IN SEVEN PARTS.        0.348           0.285           0.413               True      0.503             287
                                                                CCCI THE DAFFODILS        0.305           0.210           0.399               True      0.266             109
CCCXXXVIII ODE ON INTIMATIONS OF IMMORTALITY FROM RECOLLECTIONS OF EARLY CHILDHOOD        0.296           0.235           0.360               True      0.338             263
                                                            TA

alt.HConcatChart(...)

## 3. Authors in motion

Poems are our primary unit, but many questions are authorial: *is Milton rising? Is Pope declining? Is Shakespeare flat?* This section aggregates the Bayesian poem-level posteriors to the author level, with three views at increasing specificity: a ranking (§3.1), a heatmap (§3.2), and a full-posterior comparison for the top authors by volume (§3.3).

### 3.1 Ranking by posterior-weighted mean slope

For each author with at least three poems in the modelled set we compute a *posterior-weighted mean slope* — the mean of each author's poem slopes, weighted by lifetime quoted works, taken per MCMC draw and then summarised with 2.5th / 97.5th percentiles. Error bars therefore come from the posterior itself rather than from a classical sampling distribution. Faded bars indicate authors whose credible interval crosses zero: there is not enough evidence to say they are rising or falling.

In [22]:
if not RUN_BAMBI or "_bambi_idata" not in dir():
    print("Model bundle not loaded — run the setup cell at the top of this notebook.")
else:
    _slope_draws = _posterior[_slope_var].stack(draws=("chain", "draw"))
    _slope_draws_vals = np.asarray(_slope_draws.values)
    _pid_coord_draws = list(_slope_draws[_poem_dim].values)
    if _slope_draws_vals.shape[0] != len(_pid_coord_draws):
        _slope_draws_vals = _slope_draws_vals.T

    _pid_to_author = dict(zip(_bambi_df["poem_id"], _bambi_df["author"]))
    _pid_to_works = dict(zip(
        _bambi_df.groupby("poem_id")["n_works"].sum().index,
        _bambi_df.groupby("poem_id")["n_works"].sum().values,
    ))
    _pid_to_life = dict(zip(
        _prev_model_df.groupby("poem_id")["lifetime_works"].first().index,
        _prev_model_df.groupby("poem_id")["lifetime_works"].first().values,
    ))

    _authors_unique = sorted(set(_pid_to_author.get(pid, "Unknown") for pid in _pid_coord_draws))
    _author_poem_idx = {a: [] for a in _authors_unique}
    _author_poem_weights = {a: [] for a in _authors_unique}
    for _i, _pid in enumerate(_pid_coord_draws):
        _a = _pid_to_author.get(_pid, "Unknown")
        _author_poem_idx[_a].append(_i)
        _author_poem_weights[_a].append(float(_pid_to_life.get(_pid, 1.0)))

    _author_rows = []
    for _a in _authors_unique:
        _idxs = _author_poem_idx[_a]
        _weights = np.asarray(_author_poem_weights[_a])
        if len(_idxs) == 0 or _weights.sum() == 0:
            continue
        _slope_matrix = _slope_draws_vals[_idxs, :]
        _weighted_mean_draws = (_slope_matrix * _weights[:, None]).sum(axis=0) / _weights.sum()
        _author_rows.append({
            "author": _a,
            "n_poems": len(_idxs),
            "total_works": int(_weights.sum()),
            "weighted_slope_bayes": float(_weighted_mean_draws.mean()),
            "weighted_slope_lo": float(np.percentile(_weighted_mean_draws, 2.5)),
            "weighted_slope_hi": float(np.percentile(_weighted_mean_draws, 97.5)),
        })
    _author_bayes = (
        pd.DataFrame(_author_rows)
        .sort_values("weighted_slope_bayes", ascending=False)
        .reset_index(drop=True)
    )
    _author_bayes["significant"] = (
        (_author_bayes["weighted_slope_lo"] > 0) | (_author_bayes["weighted_slope_hi"] < 0)
    )

    AUTHOR_MIN_N_POEMS_BAYES = 3
    _author_bayes_reliable = _author_bayes[
        _author_bayes["n_poems"] >= AUTHOR_MIN_N_POEMS_BAYES
    ].copy()

    _disp_b_cols = [
        "author", "n_poems", "total_works",
        "weighted_slope_bayes", "weighted_slope_lo", "weighted_slope_hi", "significant",
    ]
    print(f"Bayesian author-level ranking ({len(_author_bayes_reliable)} authors with ≥{AUTHOR_MIN_N_POEMS_BAYES} poems):")
    print("\nTop 12 rising (weighted posterior-mean slope):")
    print(
        _author_bayes_reliable.head(12)[_disp_b_cols]
        .assign(
            weighted_slope_bayes=lambda d: d["weighted_slope_bayes"].round(3),
            weighted_slope_lo=lambda d: d["weighted_slope_lo"].round(3),
            weighted_slope_hi=lambda d: d["weighted_slope_hi"].round(3),
        )
        .to_string(index=False)
    )
    print("\nTop 12 declining:")
    print(
        _author_bayes_reliable.tail(12).sort_values("weighted_slope_bayes")[_disp_b_cols]
        .assign(
            weighted_slope_bayes=lambda d: d["weighted_slope_bayes"].round(3),
            weighted_slope_lo=lambda d: d["weighted_slope_lo"].round(3),
            weighted_slope_hi=lambda d: d["weighted_slope_hi"].round(3),
        )
        .to_string(index=False)
    )

    _auth_top_b = pd.concat([
        _author_bayes_reliable.head(12).assign(group="Rising"),
        _author_bayes_reliable.tail(12).assign(group="Declining"),
    ])
    _auth_top_b["author_short"] = _auth_top_b["author"].str[:40]
    _auth_top_b = _auth_top_b.sort_values("weighted_slope_bayes")

    _auth_chart_b = (
        alt.Chart(_auth_top_b)
        .mark_bar()
        .encode(
            x=alt.X("weighted_slope_bayes:Q", title="Weighted Bayesian slope (log-odds per decade)"),
            y=alt.Y("author_short:N", sort=None, title=None, axis=alt.Axis(labelLimit=300)),
            color=alt.condition(
                alt.datum.weighted_slope_bayes > 0,
                alt.value("#c45200"), alt.value("#154e8a"),
            ),
            opacity=alt.condition(alt.datum.significant, alt.value(1.0), alt.value(0.4)),
            tooltip=[
                "author", "n_poems", "total_works",
                alt.Tooltip("weighted_slope_bayes:Q", format=".3f"),
                alt.Tooltip("weighted_slope_lo:Q", format=".3f"),
                alt.Tooltip("weighted_slope_hi:Q", format=".3f"),
                "significant",
            ],
        )
        .properties(
            width=500, height=max(16 * len(_auth_top_b), 300),
            title=alt.TitleParams(
                f"Bayesian author-level ranking — top rising and declining poets (≥{AUTHOR_MIN_N_POEMS_BAYES} poems)",
                subtitle="Weighted over poems by lifetime_works; 95% CI from posterior | Faded bars = CI crosses 0",
            ),
        )
    )
    _auth_err_b = (
        alt.Chart(_auth_top_b)
        .mark_rule(strokeWidth=1.3, color="black")
        .encode(
            x="weighted_slope_lo:Q", x2="weighted_slope_hi:Q",
            y=alt.Y("author_short:N", sort=None),
        )
    )
    _xref_auth = (
        alt.Chart(pd.DataFrame({"x": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], opacity=0.55)
        .encode(x="x:Q")
    )
    from IPython.display import display as _display
    _display(alt.layer(_auth_chart_b, _auth_err_b, _xref_auth))

Bayesian author-level ranking (79 authors with ≥3 poems):

Top 12 rising (weighted posterior-mean slope):
                          author  n_poems  total_works  weighted_slope_bayes  weighted_slope_lo  weighted_slope_hi  significant
         Samuel Taylor Coleridge       13         1083                 0.178              0.152              0.205         True
              William Wordsworth       23         1959                 0.163              0.143              0.183         True
               Walter, Sir Scott        7         1492                 0.127              0.105              0.148         True
                William Langland        3          239                 0.115              0.076              0.155         True
      D. M. (David Macbeth) Moir        3          184                 0.110              0.051              0.170         True
                Geoffrey Chaucer       18         1792                 0.101              0.085              0.117         Tru

alt.LayerChart(...)

### 3.2 Author × decade heatmap

The poem-level heatmap in §1.4 is information-rich but hard to scan because most row labels are unfamiliar poem titles. The author-level heatmap aggregates the Bayesian fitted prevalence across all poems by the same author, replacing rows of poems with rows of authors. Fewer rows, recognisable names, same visual grammar:

- **Rows** sorted by the decade in which the author's mean fitted prevalence peaks.
- **Colour** — log-odds deviation from the cross-author baseline (average author in that decade).
- **Red** above baseline, **blue** below, **white** at baseline.

Reads top-to-bottom as a compressed canon-shift story.

In [23]:
if not RUN_BAMBI or "_plot_df_b" not in dir():
    print("Bambi small-multiples (§1.1) must run first — required for _plot_df_b.")
else:
    from IPython.display import display as _display

    # `_plot_df_b` was built from `_bambi_df.copy()` in §1.1, which already carries
    # the `author` column. Re-merging would collide (author_x / author_y). Guard
    # against both cases so this cell is robust to re-runs.
    _auth_pool = _plot_df_b.copy()
    if "author" not in _auth_pool.columns:
        _author_map = _bambi_df[["poem_id", "author"]].drop_duplicates()
        _auth_pool = _auth_pool.merge(_author_map, on="poem_id", how="left")
    _auth_pool["author"] = _auth_pool["author"].fillna("Unknown")

    _auth_poem_counts = _auth_pool.groupby("author")["poem_id"].nunique()
    AUTHOR_HEATMAP_MIN_POEMS = 3
    _qualified_authors = set(
        _auth_poem_counts[_auth_poem_counts >= AUTHOR_HEATMAP_MIN_POEMS].index
    )
    print(f"Authors with ≥{AUTHOR_HEATMAP_MIN_POEMS} poems in the modelled set: {len(_qualified_authors)}")
    _auth_pool = _auth_pool[_auth_pool["author"].isin(_qualified_authors)].copy()

    _auth_heat = (
        _auth_pool.groupby(["author", "period"], as_index=False)
        .agg(p_author=("p_bayes", "mean"), n_poems=("poem_id", "nunique"),
             total_works=("n_works", "sum"))
    )
    _auth_baseline = (
        _auth_heat.groupby("period", as_index=False)
        .agg(baseline_p=("p_author", "mean"))
    )
    _auth_heat = _auth_heat.merge(_auth_baseline, on="period", how="left")
    _auth_heat["log_odds_dev"] = (
        _safe_logit(_auth_heat["p_author"]) - _safe_logit(_auth_heat["baseline_p"])
    )

    _peak_by_author = (
        _auth_heat.loc[_auth_heat.groupby("author")["p_author"].idxmax(), ["author", "period"]]
        .rename(columns={"period": "peak_period"})
    )
    _auth_heat = _auth_heat.merge(_peak_by_author, on="author", how="left")
    _auth_totals = (
        _auth_heat.groupby("author", as_index=False)
        .agg(total_poems=("n_poems", "max"), peak_period=("peak_period", "first"))
    )

    _sort_authors = (
        _auth_totals.sort_values(["peak_period", "total_poems"], ascending=[True, False])
        ["author"].tolist()
    )
    _auth_heat["author_short"] = _auth_heat["author"].str[:40]

    _lod_cap_a = float(_auth_heat["log_odds_dev"].abs().quantile(0.95))
    _lod_cap_a = max(_lod_cap_a, 0.5)

    _auth_heatmap_chart = (
        alt.Chart(_auth_heat)
        .mark_rect()
        .encode(
            x=alt.X("period:O", title="Decade", axis=alt.Axis(labelAngle=-45)),
            y=alt.Y("author_short:N", title=None, sort=_sort_authors,
                     axis=alt.Axis(labelLimit=300)),
            color=alt.Color(
                "log_odds_dev:Q",
                title="Log-odds vs cross-author baseline",
                scale=alt.Scale(scheme="redblue", domainMid=0, domain=[-_lod_cap_a, _lod_cap_a]),
            ),
            tooltip=[
                "author",
                "period:O",
                alt.Tooltip("p_author:Q", format=".4f", title="mean fitted P"),
                alt.Tooltip("baseline_p:Q", format=".4f", title="author-baseline P"),
                alt.Tooltip("log_odds_dev:Q", format=".2f", title="log-odds dev"),
                "n_poems:Q",
            ],
        )
        .properties(
            width=560,
            height=max(22 * len(_sort_authors), 320),
            title=alt.TitleParams(
                f"Author × decade prevalence heatmap — authors with ≥{AUTHOR_HEATMAP_MIN_POEMS} poems (Bayesian)",
                subtitle="Sorted by decade of peak author-level prevalence.",
            ),
        )
    )
    _display(_auth_heatmap_chart)


Authors with ≥3 poems in the modelled set: 79


alt.Chart(...)

### 3.3 Posterior comparison for the top-five authors

A point estimate is not enough to answer *"is author A really rising faster than author B?"* Two authors with similar mean slopes may have wildly different posteriors — one tightly concentrated, the other diffuse. Here we plot the *full posterior distribution* of the weighted slope for the five most-quoted authors by total works. Where two density curves overlap heavily, we cannot distinguish their slopes. Where they sit well apart, we can.

The dashed vertical line at zero marks the "no change" reference. Authors whose density mass sits entirely to the right of zero are rising; entirely to the left, declining; straddling zero, not distinguishable from a flat trajectory.

In [24]:
if "_slope_draws_vals" not in dir() or "_author_bayes" not in dir():
    print("§3.1 (author ranking) must run first.")
else:
    from IPython.display import display as _display

    TOP_AUTHORS_N = 5
    _top_authors_by_works = (
        _author_bayes[_author_bayes["n_poems"] >= 3]
        .sort_values("total_works", ascending=False)
        .head(TOP_AUTHORS_N)["author"]
        .tolist()
    )
    print(f"Top {TOP_AUTHORS_N} authors by total lifetime works: {_top_authors_by_works}")

    _pid_to_author = dict(zip(_bambi_df["poem_id"], _bambi_df["author"]))
    _pid_to_life = dict(
        zip(
            _prev_model_df.groupby("poem_id")["lifetime_works"].first().index,
            _prev_model_df.groupby("poem_id")["lifetime_works"].first().values,
        )
    )

    _density_rows = []
    for _auth in _top_authors_by_works:
        _idxs = [i for i, pid in enumerate(_pid_coord_draws) if _pid_to_author.get(pid) == _auth]
        if not _idxs:
            continue
        _weights = np.array(
            [float(_pid_to_life.get(_pid_coord_draws[i], 1.0)) for i in _idxs]
        )
        _slopes_mat = _slope_draws_vals[_idxs, :]
        _weighted_draws = (_slopes_mat * _weights[:, None]).sum(axis=0) / _weights.sum()
        for _val in _weighted_draws:
            _density_rows.append({"author": _auth, "slope_draw": float(_val)})

    _density_df = pd.DataFrame(_density_rows)
    print(f"\nPosterior draws collected: {len(_density_df):,} rows "
          f"({_density_df['author'].nunique()} authors × {len(_density_df) // max(1, _density_df['author'].nunique()):,} draws each)")

    _density_summary = (
        _density_df.groupby("author", as_index=False)
        .agg(
            post_mean=("slope_draw", "mean"),
            post_lo=("slope_draw", lambda s: float(np.percentile(s, 2.5))),
            post_hi=("slope_draw", lambda s: float(np.percentile(s, 97.5))),
        )
        .sort_values("post_mean", ascending=False)
    )
    print("\nPosterior summaries for top authors (weighted slope, log-odds per decade):")
    print(
        _density_summary.assign(
            post_mean=lambda d: d["post_mean"].round(3),
            post_lo=lambda d: d["post_lo"].round(3),
            post_hi=lambda d: d["post_hi"].round(3),
        ).to_string(index=False)
    )

    _density_chart = (
        alt.Chart(_density_df)
        .transform_density(
            "slope_draw",
            as_=["slope", "density"],
            groupby=["author"],
        )
        .mark_area(opacity=0.45, strokeWidth=1.2)
        .encode(
            x=alt.X("slope:Q", title="Weighted Bayesian slope (log-odds per decade)"),
            y=alt.Y("density:Q", title="Posterior density"),
            color=alt.Color("author:N", title="Author", scale=alt.Scale(scheme="tableau10")),
            stroke=alt.Stroke("author:N", legend=None),
        )
        .properties(
            width=640, height=320,
            title=alt.TitleParams(
                f"Posterior distribution of weighted slope — top {TOP_AUTHORS_N} authors by total works",
                subtitle="Overlapping densities = indistinguishable trends | well-separated = credibly different",
            ),
        )
    )
    _zero_rule = (
        alt.Chart(pd.DataFrame({"x": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], opacity=0.55)
        .encode(x="x:Q")
    )
    _display(alt.layer(_density_chart, _zero_rule))


Top 5 authors by total lifetime works: ['William Shakespeare', 'Alexander Pope', 'John Milton', 'King James Bible', 'George Gordon Byron, Baron Byron']

Posterior draws collected: 40,000 rows (5 authors × 8,000 draws each)

Posterior summaries for top authors (weighted slope, log-odds per decade):
                          author  post_mean  post_lo  post_hi
George Gordon Byron, Baron Byron      0.073    0.055    0.091
             William Shakespeare      0.040    0.032    0.047
                     John Milton     -0.004   -0.014    0.007
                King James Bible     -0.007   -0.018    0.005
                  Alexander Pope     -0.060   -0.070   -0.051


alt.LayerChart(...)

## 4. Literary periods in motion

Authors cluster into literary periods (Tudor, Restoration, Augustan, Romantic, Victorian, and so on), and the Chadwyck-Healey catalogue carries this classification for every poem it covers. *Which eras, as classified by the catalogue, are rising and which are falling in the PPA record across the window?* We attach the period label to each support-qualified poem and compute the mean Bayesian slope per period, along with a credible interval.

This is a genuinely different unit from §2 and §3. A period can rise even when most of its individual authors are flat (if a few authors rise sharply), or fall even when most of its poems hold steady (if the few high-volume ones decline). The plot below is therefore a compact check on whether the poem- and author-level stories aggregate to a coherent era-level narrative.

In [25]:
if "_compare_df" not in dir():
    print("The setup cell (model bundle) must run first.")
else:
    from IPython.display import display as _display

    PERIOD_MIN_N_POEMS = 20
    SAVE_PERIOD_FIG = True

    _poetry_meta_path = POETRY_METADATA_PATH
    _ch_meta = pl.read_csv(
        str(_poetry_meta_path),
        infer_schema_length=50000,
        ignore_errors=True,
    )

    _ch_period = (
        _ch_meta.select(
            pl.col("title_id").alias("poem_id"),
            pl.col("period")
            .cast(pl.Utf8, strict=False)
            .fill_null("")
            .str.strip_chars()
            .alias("ch_period"),
        )
        .filter(pl.col("ch_period") != "")
        .filter(pl.col("ch_period") != "None")
        .unique(subset=["poem_id"])
        .to_pandas()
    )
    print(f"Chadwyck period data: {len(_ch_period):,} poems with non-null period")

    _period_slope_b = _compare_df[
        [
            "poem_id",
            "poem_label",
            "slope_bayes",
            "slope_bayes_lo",
            "slope_bayes_hi",
            "lifetime_works",
        ]
    ].merge(_ch_period, on="poem_id", how="left")

    _period_slope_b["bayes_sig"] = (
        (_period_slope_b["slope_bayes_lo"] > 0)
        | (_period_slope_b["slope_bayes_hi"] < 0)
    )

    _has_period_b = _period_slope_b["ch_period"].notna()
    print(f"Support-qualified poems with period: {_has_period_b.sum()}/{len(_period_slope_b)}")

    _period_agg_b = (
        _period_slope_b[_has_period_b]
        .groupby("ch_period", as_index=False)
        .agg(
            n_poems=("poem_id", "nunique"),
            mean_slope=("slope_bayes", "mean"),
            median_slope=("slope_bayes", "median"),
            q25_slope=("slope_bayes", lambda s: s.quantile(0.25)),
            q75_slope=("slope_bayes", lambda s: s.quantile(0.75)),
            frac_significant=("bayes_sig", "mean"),
            mean_works=("lifetime_works", "mean"),
            total_works=("lifetime_works", "sum"),
        )
        .sort_values("mean_slope", ascending=False)
        .reset_index(drop=True)
    )

    _period_agg_b["period_short"] = _period_agg_b["ch_period"].str[:45]

    _period_agg_plot_b = (
        _period_agg_b
        .loc[_period_agg_b["n_poems"] >= PERIOD_MIN_N_POEMS]
        .copy()
        .sort_values("mean_slope", ascending=False)
        .reset_index(drop=True)
    )

    _period_agg_plot_b["n_label"] = (
        "n=" + _period_agg_plot_b["n_poems"].astype(int).astype(str)
    )

    _x_abs_max = float(
        _period_agg_plot_b[["mean_slope", "q25_slope", "q75_slope"]]
        .abs()
        .max()
        .max()
    )
    _x_pad = max(0.006, _x_abs_max * 0.06)

    _period_agg_plot_b["label_x"] = _period_agg_plot_b.apply(
        lambda r: (
            r["q75_slope"] + _x_pad
            if r["mean_slope"] >= 0
            else r["q25_slope"] - _x_pad
        ),
        axis=1,
    )

    _x_domain = [-(_x_abs_max * 1.35), _x_abs_max * 1.35]

    _dropped_periods_b = (
        _period_agg_b
        .loc[_period_agg_b["n_poems"] < PERIOD_MIN_N_POEMS]
        .copy()
        .sort_values("n_poems", ascending=False)
    )

    print(f"\nMean Bayesian slope by Chadwyck literary period (n >= {PERIOD_MIN_N_POEMS} poems):")
    print(
        _period_agg_plot_b[
            [
                "period_short",
                "n_poems",
                "mean_slope",
                "median_slope",
                "q25_slope",
                "q75_slope",
                "frac_significant",
                "mean_works",
            ]
        ]
        .assign(
            mean_slope=lambda d: d["mean_slope"].round(3),
            median_slope=lambda d: d["median_slope"].round(3),
            q25_slope=lambda d: d["q25_slope"].round(3),
            q75_slope=lambda d: d["q75_slope"].round(3),
            frac_significant=lambda d: d["frac_significant"].round(2),
            mean_works=lambda d: d["mean_works"].round(0),
        )
        .to_string(index=False)
    )

    if len(_dropped_periods_b) > 0:
        print(f"\nDropped from plot because n < {PERIOD_MIN_N_POEMS}:")
        print(
            _dropped_periods_b[
                ["period_short", "n_poems", "mean_slope", "median_slope"]
            ]
            .assign(
                mean_slope=lambda d: d["mean_slope"].round(3),
                median_slope=lambda d: d["median_slope"].round(3),
            )
            .to_string(index=False)
        )

    _base = alt.Chart(_period_agg_plot_b)

    _bars = (
        _base
        .mark_bar()
        .encode(
            x=alt.X(
                "mean_slope:Q",
                title="Mean poem-level Bayesian slope (log-odds per decade)",
                scale=alt.Scale(domain=_x_domain),
            ),
            y=alt.Y(
                "period_short:N",
                sort=None,
                title=None,
                axis=alt.Axis(labelLimit=340),
            ),
            color=alt.condition(
                alt.datum.mean_slope > 0,
                alt.value("#c45200"),
                alt.value("#154e8a"),
            ),
            tooltip=[
                alt.Tooltip("ch_period:N", title="Chadwyck period"),
                alt.Tooltip("n_poems:Q", title="Poems"),
                alt.Tooltip("mean_slope:Q", title="Mean slope", format=".3f"),
                alt.Tooltip("median_slope:Q", title="Median slope", format=".3f"),
                alt.Tooltip("q25_slope:Q", title="25th percentile", format=".3f"),
                alt.Tooltip("q75_slope:Q", title="75th percentile", format=".3f"),
                alt.Tooltip("frac_significant:Q", title="Share individually significant", format=".0%"),
                alt.Tooltip("mean_works:Q", title="Mean lifetime works", format=".0f"),
                alt.Tooltip("total_works:Q", title="Total lifetime works", format=".0f"),
            ],
        )
    )

    _zero_rule = (
        alt.Chart(pd.DataFrame({"x": [0.0]}))
        .mark_rule(color="black", strokeDash=[4, 3], opacity=0.65)
        .encode(x=alt.X("x:Q", scale=alt.Scale(domain=_x_domain)))
    )

    _iqr_whisker = (
        _base
        .mark_rule(color="black", strokeWidth=1.4)
        .encode(
            x=alt.X("q25_slope:Q", scale=alt.Scale(domain=_x_domain)),
            x2="q75_slope:Q",
            y=alt.Y("period_short:N", sort=None),
        )
    )

    _median_dot = (
        _base
        .mark_point(color="black", filled=True, size=34)
        .encode(
            x=alt.X("median_slope:Q", scale=alt.Scale(domain=_x_domain)),
            y=alt.Y("period_short:N", sort=None),
        )
    )

    _n_labels = (
        _base
        .mark_text(
            color="#555",
            fontSize=10,
            baseline="middle",
            align="center",
        )
        .encode(
            x=alt.X("label_x:Q", scale=alt.Scale(domain=_x_domain)),
            y=alt.Y("period_short:N", sort=None),
            text="n_label:N",
        )
    )

    _period_chart_b = (
        alt.layer(_zero_rule, _bars, _iqr_whisker, _median_dot, _n_labels)
        .properties(
            width=580,
            height=max(30 * len(_period_agg_plot_b), 270),
            title=alt.TitleParams(
                f"Mean poem-level prevalence trend by Chadwyck period (n >= {PERIOD_MIN_N_POEMS})",
                subtitle=(
                    "Bars = mean posterior-mean poem slope; black whiskers = interquartile range; "
                    "black dots = median; labels show modeled poems per period"
                ),
            ),
        )
        .configure_view(strokeWidth=0)
    )

    if SAVE_PERIOD_FIG:
        _FIG_DIR = FIGURES_DIR
        _FIG_DIR.mkdir(parents=True, exist_ok=True)

        _period_chart_b.save(str(_FIG_DIR / "period_mean_bayesian_slopes_n20.html"))
        _period_chart_b.save(str(_FIG_DIR / "period_mean_bayesian_slopes_n20.svg"))
        try:
            _period_chart_b.save(str(_FIG_DIR / "period_mean_bayesian_slopes_n20.pdf"))
        except Exception as _e:
            print("PDF save skipped:", _e)

        print(f"\nSaved period figure to {_FIG_DIR}")

    _display(_period_chart_b)

Chadwyck period data: 311,253 poems with non-null period
Support-qualified poems with period: 839/906

Mean Bayesian slope by Chadwyck literary period (n >= 20 poems):
                          period_short  n_poems  mean_slope  median_slope  q25_slope  q75_slope  frac_significant  mean_works
       Middle English Poetry 1100-1400       23       0.092         0.105      0.053      0.115              0.61        79.0
Miscellanies and Collections 1550-1900       92       0.039         0.041     -0.017      0.100              0.52        87.0
    Early Nineteenth-Century 1800-1834      209       0.029         0.027     -0.019      0.083              0.31        88.0
                       Tudor 1500-1580       43       0.025         0.031     -0.014      0.070              0.28        58.0
      Mid Nineteenth-Century 1835-1869       59       0.013         0.005     -0.040      0.066              0.24        84.0
       Jacobean and Caroline 1603-1660       65       0.003         0.006   

alt.LayerChart(...)

In [26]:
# ----------------------------------------------------------------------------
# Period-level ridgeplot from the Bayesian model
# Stable version: one chart per period, then vconcat.
# ----------------------------------------------------------------------------

if "_period_slope_b" in dir() and not _period_slope_b.empty:
    from IPython.display import display as _display

    RIDGE_MIN_POEMS = 20
    SAVE_RIDGE_FIG = True

    EXCLUDE_PERIODS = {
        "Miscellanies and Collections 1550-1900",
        "Mid Nineteenth-Century 1835-1869",
    }

    _ridge_data_b = _period_slope_b.dropna(subset=["ch_period"]).copy()
    _ridge_data_b = _ridge_data_b[
        ~_ridge_data_b["ch_period"].isin(EXCLUDE_PERIODS)
    ].copy()

    _period_counts_b = _ridge_data_b.groupby("ch_period", observed=True).size()
    _ridge_periods_b = _period_counts_b[_period_counts_b >= RIDGE_MIN_POEMS].index.tolist()
    _ridge_data_b = _ridge_data_b[_ridge_data_b["ch_period"].isin(_ridge_periods_b)].copy()

    def _short_period_b(name):
        replacements = {
            "Middle English Poetry 1100-1400": "Middle English (1100–1400)",
            "Tudor 1500-1580": "Tudor (1500–1580)",
            "Jacobean and Caroline 1603-1660": "Jacobean/Caroline (1603–1660)",
            "Restoration 1660-1700": "Restoration (1660–1700)",
            "Early Eighteenth-Century 1700-1749": "Early eighteenth century (1700–1749)",
            "Later Eighteenth-Century 1750-1799": "Later eighteenth century (1750–1799)",
            "Early Nineteenth-Century 1800-1834": "Early nineteenth century (1800–1834)",
            "Mid Nineteenth-Century 1835-1869": "Mid nineteenth century (1835–1869)",
            "Later Nineteenth-Century 1870-1899": "Later nineteenth century (1870–1899)",
            "Twentieth-Century 1900-1999": "Twentieth century (1900–1999)",
            "Middle English Lyrics and Ballads 1100-1500": "Middle English lyrics/ballads (1100–1500)",
            "Middle English Romances 1100-1500": "Middle English romances (1100–1500)",
            "Fifteenth-Century Poetry": "Fifteenth-century poetry",
            "Anglo-Saxon Poetry 600-1100": "Anglo-Saxon poetry (600–1100)",
            "Emblems, Epigrams, Formal Satires 1500-1700": "Emblems/epigrams/satires (1500–1700)",
            "Miscellanies and Collections 1550-1900": "Miscellanies and collections (1550–1900)",
        }
        return replacements.get(name, name)

    _period_stats_b = (
        _ridge_data_b.groupby("ch_period", observed=True)
        .agg(
            n_poems=("poem_id", "nunique"),
            median_slope=("slope_bayes", "median"),
            mean_slope=("slope_bayes", "mean"),
            q25_slope=("slope_bayes", lambda s: s.quantile(0.25)),
            q75_slope=("slope_bayes", lambda s: s.quantile(0.75)),
            frac_significant=("bayes_sig", "mean"),
        )
        .reset_index()
        .sort_values("median_slope", ascending=False)
    )

    _period_stats_b["period_base"] = _period_stats_b["ch_period"].apply(_short_period_b)
    _period_stats_b["period_label"] = (
        _period_stats_b["period_base"]
        + ", n="
        + _period_stats_b["n_poems"].astype(int).astype(str)
    )

    _RIDGE_X_DOMAIN = [-0.35, 0.35]
    _RIDGE_BANDWIDTH = 0.020

    _ridge_rows = []

    for _i, _stat in _period_stats_b.reset_index(drop=True).iterrows():
        _period = _stat["ch_period"]
        _label = _stat["period_label"]
        _median = float(_stat["median_slope"])
        _mean = float(_stat["mean_slope"])
        _n = int(_stat["n_poems"])

        _sub = _ridge_data_b[_ridge_data_b["ch_period"] == _period].copy()
        _sub["slope_clipped"] = _sub["slope_bayes"].clip(
            _RIDGE_X_DOMAIN[0],
            _RIDGE_X_DOMAIN[1],
        )

        _color = "#c45200" if _median > 0 else "#154e8a"
        _opacity = min(0.90, 0.35 + abs(_median) / 0.18)

        _density = (
            alt.Chart(_sub)
            .transform_density(
                "slope_clipped",
                extent=_RIDGE_X_DOMAIN,
                steps=240,
                as_=["slope", "density"],
                bandwidth=_RIDGE_BANDWIDTH,
            )
            .mark_area(
                interpolate="monotone",
                fill=_color,
                fillOpacity=_opacity,
                stroke="white",
                strokeWidth=0.7,
            )
            .encode(
                x=alt.X(
                    "slope:Q",
                    title=None if _i < len(_period_stats_b) - 1 else
                    "Posterior-mean poem slope, log-odds per decade (Declining ← 0 → Rising)",
                    scale=alt.Scale(domain=_RIDGE_X_DOMAIN),
                    axis=alt.Axis(
                        grid=False,
                        values=[-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3],
                        labels=(_i == len(_period_stats_b) - 1),
                        ticks=(_i == len(_period_stats_b) - 1),
                    ),
                ),
                y=alt.Y("density:Q", title=None, axis=None),
                tooltip=[
                    alt.Tooltip("slope_clipped:Q", title="Poem slope", format=".3f"),
                ],
            )
        )

        _zero = (
            alt.Chart(pd.DataFrame({"slope": [0.0]}))
            .mark_rule(color="black", strokeDash=[4, 3], opacity=0.45)
            .encode(x=alt.X("slope:Q", scale=alt.Scale(domain=_RIDGE_X_DOMAIN)))
        )

        _median_rule = (
            alt.Chart(pd.DataFrame({"slope": [_median]}))
            .mark_rule(color="black", opacity=0.55, strokeWidth=1.0)
            .encode(x=alt.X("slope:Q", scale=alt.Scale(domain=_RIDGE_X_DOMAIN)))
        )

        _label_chart = (
            alt.Chart(pd.DataFrame({
                "x": [_RIDGE_X_DOMAIN[0]],
                "y": [0],
                "label": [_label],
            }))
            .mark_text(
                align="right",
                baseline="middle",
                dx=-8,
                fontSize=11,
                color="black",
            )
            .encode(
                x=alt.X("x:Q", scale=alt.Scale(domain=_RIDGE_X_DOMAIN)),
                y=alt.value(22),
                text="label:N",
            )
        )

        _row = (
            alt.layer(_zero, _density, _median_rule, _label_chart)
            .properties(width=650, height=46)
        )

        _ridge_rows.append(_row)

    _title = "Poem-Level Slope Distributions by Literary Period"
    #_subtitle = (
    #    "Each ridge shows the distribution of posterior-mean poem slopes within a Chadwyck-Healey period. "
    #    f"Rows are sorted by median slope; periods with fewer than {RIDGE_MIN_POEMS} modeled poems are omitted"
    #    + ("; miscellanies and collections are excluded." if EXCLUDE_MISCELLANIES else ".")
    #)

    _ridge_b = (
        alt.vconcat(*_ridge_rows, spacing=-10)
        .properties(
            title=alt.TitleParams(
                text=_title,
                #subtitle=_subtitle,
                fontSize=15,
                subtitleFontSize=11,
                subtitleColor="#555",
                anchor="start",
            )
        )
        .configure_view(stroke=None)
        .configure_axis(labelFontSize=10, titleFontSize=11)
    )

    _display(_ridge_b)

    print(f"\nRidgeplot includes {len(_ridge_periods_b)} periods with >= {RIDGE_MIN_POEMS} poems each.")
    print(
        _period_stats_b[
            ["period_base", "n_poems", "median_slope", "mean_slope", "q25_slope", "q75_slope", "frac_significant"]
        ]
        .assign(
            median_slope=lambda d: d["median_slope"].round(3),
            mean_slope=lambda d: d["mean_slope"].round(3),
            q25_slope=lambda d: d["q25_slope"].round(3),
            q75_slope=lambda d: d["q75_slope"].round(3),
            frac_significant=lambda d: d["frac_significant"].round(2),
        )
        .to_string(index=False)
    )

    _excluded_low_n = _period_counts_b[_period_counts_b < RIDGE_MIN_POEMS]
    if len(_excluded_low_n) > 0:
        print(f"\nPeriods excluded for low sample size: {_excluded_low_n.to_dict()}")

    if SAVE_RIDGE_FIG:
        _FIG_DIR = FIGURES_DIR
        _FIG_DIR.mkdir(parents=True, exist_ok=True)

        _ridge_b.save(str(_FIG_DIR / "period_slope_ridgeplot.html"))
        _ridge_b.save(str(_FIG_DIR / "period_slope_ridgeplot.svg"))
        try:
            _ridge_b.save(str(_FIG_DIR / "period_slope_ridgeplot.pdf"))
        except Exception as _e:
            print("PDF save skipped:", _e)

        print(f"\nSaved ridgeplot to {_FIG_DIR}")

else:
    print("§4 must run first — _period_slope_b is required for the ridgeplot.")

alt.VConcatChart(...)


Ridgeplot includes 7 periods with >= 20 poems each.
                         period_base  n_poems  median_slope  mean_slope  q25_slope  q75_slope  frac_significant
          Middle English (1100–1400)       23         0.105       0.092      0.053      0.115              0.61
                   Tudor (1500–1580)       43         0.031       0.025     -0.014      0.070              0.28
Early nineteenth century (1800–1834)      209         0.027       0.029     -0.019      0.083              0.31
       Jacobean/Caroline (1603–1660)       65         0.006       0.003     -0.057      0.055              0.42
Later eighteenth century (1750–1799)      157         0.003       0.001     -0.044      0.048              0.32
Early eighteenth century (1700–1749)      114        -0.075      -0.072     -0.113     -0.038              0.73
             Restoration (1660–1700)       67        -0.124      -0.117     -0.177     -0.055              0.78

Periods excluded for low sample size: {'Anglo-Saxo